# Оценка качества эмбеддингов: голова предсказания статов матча

**Задача:** по одному матчу (эмбеддинги после трансформера) обучаем голову, которая предсказывает **2×39** командных статов (суммы по команде A и B). Один матч → один таргет. Цель — оценить, насколько эмбеддинги несут информацию о статах (не из статьи). Энкодер заморожен, обучается только голова. Сравниваем **embed_128** и **embed_64**.

In [1]:
import sys
from pathlib import Path

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pickle
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from safetensors.torch import load_file as load_safetensors

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 1. Пути и данные

In [2]:
ENCODER_CKPT_DIR = ROOT / "model_trained" / "embed_128"
ENCODER_CKPT_DIR_64 = ROOT / "model_trained" / "embed_64"
METADATA_DIR = ROOT / "notebooks" / "train_sample_processed" / "metadata"
PROCESSED_CSV = ROOT / "notebooks" / "train_sample_processed" / "processed.csv"
OUTPUT_DIR = ROOT / "notebooks" / "stats_head_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "embed_64").mkdir(parents=True, exist_ok=True)
HEAD_PATH_64 = OUTPUT_DIR / "embed_64" / "stats_head.pt"

assert ENCODER_CKPT_DIR.exists(), f"Не найден чекпоинт энкодера: {ENCODER_CKPT_DIR}"
assert ENCODER_CKPT_DIR_64.exists(), f"Не найден чекпоинт энкодера 64: {ENCODER_CKPT_DIR_64}"
assert METADATA_DIR.exists(), f"Не найден metadata: {METADATA_DIR}"
assert PROCESSED_CSV.exists(), f"Не найден processed CSV: {PROCESSED_CSV}"

In [3]:
import pandas as pd

def load_vocab(metadata_dir: Path):
    with open(metadata_dir / "player_name2id.pickle", "rb") as f:
        player_name2id = pickle.load(f)
    with open(metadata_dir / "team_name2id.pickle", "rb") as f:
        team_name2id = pickle.load(f)
    n_players = len(player_name2id)
    n_teams = len(team_name2id)
    return {
        "player_name2id": player_name2id,
        "team_name2id": team_name2id,
        "player_pad_token_id": n_players + 1,
        "team_pad_token_id": n_teams,
    }

from data.preprocessing import STAT_COLUMNS

vocab = load_vocab(METADATA_DIR)
df = pd.read_csv(PROCESSED_CSV)
print("Матчей:", df["match_id"].nunique(), "| Игроков в словаре:", len(vocab["player_name2id"]))
print("Вход: 39 статов матча → энкодер → голова → 2×39 (командные суммы)")

Матчей: 2923 | Игроков в словаре: 6393
Вход: 39 статов матча → энкодер → голова → 2×39 (командные суммы)


## 2. Датасет и коллатор

In [4]:
from data.dataset import MatchDatasetNMSP
from data.collator import DataCollatorNMSP

max_seq_length = 36
num_target_stats = 39  # 39 статов на команду → выход головы 2*39=78
ds_full = MatchDatasetNMSP(
    df,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=max_seq_length,
    player_pad_token_id=vocab["player_pad_token_id"],
    team_pad_token_id=vocab["team_pad_token_id"],
    position_pad_token_id=25,
    target_columns=STAT_COLUMNS,
)

n = len(ds_full)
n_val = max(1, int(n * 0.1))
n_train = n - n_val
train_ds = Subset(ds_full, range(0, n_train))
eval_ds = Subset(ds_full, range(n_train, n))

collator_nmsp = DataCollatorNMSP()
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collator_nmsp, num_workers=0)
eval_loader = DataLoader(eval_ds, batch_size=32, shuffle=False, collate_fn=collator_nmsp, num_workers=0)
print("Train батчей:", len(train_loader), "| Eval батчей:", len(eval_loader))

Train батчей: 83 | Eval батчей: 10


## 3. Модель: энкодер (заморожен) + NMSPHead (flatten → MLP → 2×39)

In [5]:
from models.encoder import PlayerEncoder
from models.heads import NMSPHead
from data.preprocessing import FORM_STATS_SIZE

embed_size = 128
players_vocab_size = vocab["player_pad_token_id"]
teams_vocab_size = vocab["team_pad_token_id"] + 1

encoder = PlayerEncoder(
    embed_size=embed_size,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=FORM_STATS_SIZE,
    players_vocab_size=players_vocab_size,
    teams_vocab_size=teams_vocab_size,
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)

state = load_safetensors(ENCODER_CKPT_DIR / "model.safetensors")
encoder_state = {k.replace("encoder.", ""): v for k, v in state.items() if k.startswith("encoder.")}
encoder.load_state_dict(encoder_state, strict=True)
for p in encoder.parameters():
    p.requires_grad = False

head = NMSPHead(embed_size=embed_size, max_seq_length=max_seq_length, num_target_stats=num_target_stats, hidden_dim=512)

encoder = encoder.to(device)
head = head.to(device)
print("Параметров головы:", sum(p.numel() for p in head.parameters()))

Параметров головы: 2399822


## 4. Обучение

In [6]:
from training.stats_trainer import NMSPHeadTrainer

trainer = NMSPHeadTrainer(
    encoder=encoder,
    head=head,
    train_loader=train_loader,
    eval_loader=eval_loader,
    output_dir=str(OUTPUT_DIR),
    num_epochs=20,
    lr=1e-3,
    weight_decay=0.01,
    device=device,
    logging_steps=1,
    save_best=True,
)
trainer.train()

Epoch 1/20  Train MSE: 942.282392  Eval MSE: 163.678309
Epoch 2/20  Train MSE: 144.933743  Eval MSE: 113.582358
Epoch 3/20  Train MSE: 106.632119  Eval MSE: 107.219704
Epoch 4/20  Train MSE: 89.442593  Eval MSE: 104.742195
Epoch 5/20  Train MSE: 77.183476  Eval MSE: 97.860360
Epoch 6/20  Train MSE: 70.819713  Eval MSE: 134.233268
Epoch 7/20  Train MSE: 64.482647  Eval MSE: 98.055628
Epoch 8/20  Train MSE: 61.954562  Eval MSE: 103.799087
Epoch 9/20  Train MSE: 60.972746  Eval MSE: 104.108540
Epoch 10/20  Train MSE: 60.015997  Eval MSE: 104.746468
Epoch 11/20  Train MSE: 56.148442  Eval MSE: 105.386369
Epoch 12/20  Train MSE: 52.313364  Eval MSE: 88.274471
Epoch 13/20  Train MSE: 50.584091  Eval MSE: 94.281284
Epoch 14/20  Train MSE: 48.830566  Eval MSE: 102.921806
Epoch 15/20  Train MSE: 47.897537  Eval MSE: 106.068539
Epoch 16/20  Train MSE: 47.404538  Eval MSE: 91.798758
Epoch 17/20  Train MSE: 45.775244  Eval MSE: 92.224286
Epoch 18/20  Train MSE: 45.458302  Eval MSE: 96.467735
Epoch

88.27447052001953

In [7]:
# Обучение выполнено в ячейке выше через NMSPHeadTrainer

## 4b. Обучение головы embed_64

Энкодер из `model_trained/embed_64` (64-мерные эмбеддинги). Обучаем только голову, энкодер заморожен; чекпоинт сохраняется в `stats_head_output/embed_64/stats_head.pt`.

In [8]:
encoder_64 = PlayerEncoder(
    embed_size=64,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=FORM_STATS_SIZE,
    players_vocab_size=players_vocab_size,
    teams_vocab_size=teams_vocab_size,
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
state_64 = load_safetensors(ENCODER_CKPT_DIR_64 / "model.safetensors")
enc_state_64 = {k.replace("encoder.", ""): v for k, v in state_64.items() if k.startswith("encoder.")}
encoder_64.load_state_dict(enc_state_64, strict=True)
for p in encoder_64.parameters():
    p.requires_grad = False

head_64 = NMSPHead(embed_size=64, max_seq_length=max_seq_length, num_target_stats=num_target_stats, hidden_dim=512)
encoder_64 = encoder_64.to(device)
head_64 = head_64.to(device)
print("Параметров головы 64:", sum(p.numel() for p in head_64.parameters()))

Параметров головы 64: 1220174


In [9]:
trainer_64 = NMSPHeadTrainer(
    encoder=encoder_64,
    head=head_64,
    train_loader=train_loader,
    eval_loader=eval_loader,
    output_dir=str(OUTPUT_DIR / "embed_64"),
    num_epochs=20,
    lr=1e-3,
    weight_decay=0.01,
    device=device,
    logging_steps=1,
    save_best=True,
)
trainer_64.train()

Epoch 1/20  Train MSE: 1176.612269  Eval MSE: 348.812808
Epoch 2/20  Train MSE: 229.967300  Eval MSE: 256.330836
Epoch 3/20  Train MSE: 165.061814  Eval MSE: 227.269264
Epoch 4/20  Train MSE: 137.690846  Eval MSE: 205.300861
Epoch 5/20  Train MSE: 125.085278  Eval MSE: 192.120343
Epoch 6/20  Train MSE: 112.157724  Eval MSE: 196.734811
Epoch 7/20  Train MSE: 105.413712  Eval MSE: 211.005719
Epoch 8/20  Train MSE: 99.789605  Eval MSE: 191.137642
Epoch 9/20  Train MSE: 89.751876  Eval MSE: 182.926421
Epoch 10/20  Train MSE: 86.474105  Eval MSE: 195.372031
Epoch 11/20  Train MSE: 80.967615  Eval MSE: 171.845962
Epoch 12/20  Train MSE: 80.589621  Eval MSE: 188.201163
Epoch 13/20  Train MSE: 76.594620  Eval MSE: 179.591021
Epoch 14/20  Train MSE: 72.457868  Eval MSE: 180.986950
Epoch 15/20  Train MSE: 68.523108  Eval MSE: 170.649271
Epoch 16/20  Train MSE: 70.134773  Eval MSE: 158.524985
Epoch 17/20  Train MSE: 66.828922  Eval MSE: 172.579217
Epoch 18/20  Train MSE: 66.023650  Eval MSE: 172.

158.52498474121094

## 5. Evaluation

**Таблица: средние значения по eval (все в одной шкале).**

- **Наст. значение** — среднее целевое значение статы по eval.
- **embed_128** / **embed_64** — среднее предсказание модели по eval (в тех же единицах). Жирное — предсказание ближе к настоящему значению.

In [12]:
from data.preprocessing import STAT_COLUMNS, FORM_STATS_SIZE
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

from models.encoder import PlayerEncoder
from models.heads import NMSPHead
from training.stats_trainer import NMSPHeadTrainer

# Сначала собираем target_stats по eval (после этого eval_loader исчерпан)
targets = []
for batch in eval_loader:
    targets.append(batch["target_stats"].numpy())
targets = np.concatenate(targets, axis=0)
mean_target = targets.mean(axis=0)
# Собираем средние предсказания embed_128
eval_loader = DataLoader(eval_ds, batch_size=32, shuffle=False, collate_fn=collator_nmsp, num_workers=0)
trainer.eval_loader = eval_loader
trainer.encoder.eval()
trainer.head.eval()
preds_128_list = []
with torch.no_grad():
    for batch in eval_loader:
        pred, _ = trainer._forward_batch(batch)
        preds_128_list.append(pred.cpu().numpy())
preds_128 = np.concatenate(preds_128_list, axis=0)
mean_pred128_78 = preds_128.mean(axis=0)

# Восстанавливаем eval_loader для embed_64
eval_loader = DataLoader(eval_ds, batch_size=32, shuffle=False, collate_fn=collator_nmsp, num_workers=0)

# Модель embed_64 для таблицы
encoder_64 = PlayerEncoder(
    embed_size=64,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=FORM_STATS_SIZE,
    players_vocab_size=players_vocab_size,
    teams_vocab_size=teams_vocab_size,
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
state_64 = load_safetensors(ENCODER_CKPT_DIR_64 / "model.safetensors")
enc_state_64 = {k.replace("encoder.", ""): v for k, v in state_64.items() if k.startswith("encoder.")}
encoder_64.load_state_dict(enc_state_64, strict=True)
for p in encoder_64.parameters():
    p.requires_grad = False
head_64 = NMSPHead(embed_size=64, max_seq_length=max_seq_length, num_target_stats=num_target_stats, hidden_dim=512)
if HEAD_PATH_64.exists():
    head_64.load_state_dict(torch.load(HEAD_PATH_64, map_location="cpu"))
encoder_64, head_64 = encoder_64.to(device), head_64.to(device)
trainer_64 = NMSPHeadTrainer(encoder_64, head_64, train_loader, eval_loader, output_dir=str(OUTPUT_DIR), num_epochs=1, device=device)
trainer_64.encoder.eval()
trainer_64.head.eval()
preds_64_list = []
with torch.no_grad():
    for batch in eval_loader:
        pred, _ = trainer_64._forward_batch(batch)
        preds_64_list.append(pred.cpu().numpy())
preds_64 = np.concatenate(preds_64_list, axis=0)
mean_pred64_78 = preds_64.mean(axis=0)

# 18 статов для таблицы
STAT_COLUMNS_18 = [
    "pass_total", "pass_cross", "pass_shot_assist", "pass_goal_assist", "pass_through_ball",
    "shot_total", "shot_statsbomb_xg", "shot_goal",
    "interception_won", "block_total", "clearance_total", "ball_recovery_total", "counterpress_total",
    "dribble_complete", "foul_won_total", "foul_committed_total",
    "goalkeeper_save", "shot_saved",
]
indices_18 = [STAT_COLUMNS.index(c) for c in STAT_COLUMNS_18]

# Средние по eval: таргет и предсказания (команда A и B усреднены)
mean_true_per_stat = (mean_target[:num_target_stats] + mean_target[num_target_stats:]) / 2
mean_pred128_per_stat = (mean_pred128_78[:num_target_stats] + mean_pred128_78[num_target_stats:]) / 2
mean_pred64_per_stat = (mean_pred64_78[:num_target_stats] + mean_pred64_78[num_target_stats:]) / 2

best_col_arr = []
rows = []
for i in indices_18:
    true_val = mean_true_per_stat[i]
    dist_128 = abs(mean_pred128_per_stat[i] - true_val)
    dist_64 = abs(mean_pred64_per_stat[i] - true_val)
    best_idx = int(np.argmin([dist_128, dist_64]))
    best_col_arr.append(best_idx)
    rows.append({
        "Statistics": STAT_COLUMNS[i],
        "Наст. значение": f"{mean_true_per_stat[i]:.2f}",
        "embed_128": f"{mean_pred128_per_stat[i]:.2f}",
        "embed_64": f"{mean_pred64_per_stat[i]:.2f}",
    })

df_t = pd.DataFrame(rows)

def style_table(st):
    out = pd.DataFrame("", index=st.index, columns=st.columns)
    for i in range(len(st)):
        bc = best_col_arr[i]  # 0=embed_128, 1=embed_64
        out.iloc[i, 2 + bc] = "font-weight: bold"
    return out

df_t.style.apply(style_table, axis=None)

,Statistics,Наст. значение,embed_128,embed_64
0,pass_total,465.20,56.61,72.74
1,pass_cross,10.58,4.50,5.39
2,pass_shot_assist,7.56,3.38,3.74
3,pass_goal_assist,0.67,0.94,1.10
4,pass_through_ball,1.91,2.69,2.33
5,shot_total,11.31,3.87,4.37
6,shot_statsbomb_xg,1.08,0.82,0.79
7,shot_goal,1.08,1.38,1.52
8,interception_won,3.67,2.53,2.61
9,block_total,17.84,6.19,5.33
